In [ ]:
# PACE-VCF Training Notebook
## XGBoost Full Model + Monte Carlo Feature Selection
## UPDATED: Native NaN handling (no median fill)
## UPDATED: New chip dataset with 70-15-15 split

# ---

# ## Cell 1: Imports and Configuration

# =============================================================================
# PACE-VCF Training: XGBoost + Monte Carlo Feature Selection
# UPDATED: Use XGBoost native NaN handling instead of median fill
# UPDATED: New chip dataset with 70-15-15 train/val/test split
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import rasterio
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
import logging
import warnings
import time

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================

# NEW: Chip-based training data
CHIP_BASE = Path("/explore/nobackup/people/ajkerr1/PACE_Hyperspectral/chip_outputs/arctic_fixed/date_2026_08_28-time_14_57_10/raw")
PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PACE_YEAR = 2025
TILE_SIZE_PACE = 600
NO_DATA = -10001

# Feature flags
INCLUDE_PACE_METRICS = True      # Include PACE_Metrics.tif (hyperspectral VIs)
INCLUDE_PACE_ALTSORT = True      # Include PACE_AltSort_Metrics.tif
INCLUDE_UNSORTED = False         # Include UnsortedMonthly* features (calendar-based)

# Model parameters
RANDOM_STATE = 42

# NEW: 70-15-15 split
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

# Balanced sampling parameters
TARGET_BARE_PCT = 0.25
TARGET_LOW_PCT = 0.25
TARGET_FOREST_PCT = 0.08

# XGBoost GPU parameters (full model)
XGB_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 500,
    'max_depth': 10,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbosity': 1,
    'missing': np.nan,  # Native NaN handling
}

# Monte Carlo parameters
N_TRIALS = 100
FEATURES_PER_TRIAL = 50
MIN_FEATURE_USAGE = 10
TOP_N_FEATURES = 50

# XGBoost parameters (faster for MC trials)
XGB_TRIAL_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 100,
    'max_depth': 8,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'verbosity': 0,
    'missing': np.nan,
}

# XGBoost parameters (final MC model)
XGB_FINAL_PARAMS = {
    'tree_method': 'gpu_hist',
    'n_estimators': 500,
    'max_depth': 10,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE,
    'verbosity': 1,
    'missing': np.nan,
}

print("="*70)
print("PACE-VCF TRAINING NOTEBOOK")
print("="*70)
print(f"  Data source: {CHIP_BASE}")
print(f"  Split: {TRAIN_SIZE*100:.0f}% train / {VAL_SIZE*100:.0f}% val / {TEST_SIZE*100:.0f}% test")
print(f"  PACE metrics: {INCLUDE_PACE_METRICS}")
print(f"  AltSort metrics: {INCLUDE_PACE_ALTSORT}")
print(f"  Monte Carlo trials: {N_TRIALS}")
print(f"  Top-N features: {TOP_N_FEATURES}")
print(f"  NaN handling: NATIVE (XGBoost learns missing value behavior)")
print("="*70)


# =============================================================================
# Cell 2: Data Loading Functions (COMPLETE)
# =============================================================================

def load_chip_training_data(force_extract=True):
    """
    Load training data from chip-based dataset.
    
    Structure:
        CHIP_BASE/<tile>/inputs/<tile>_r<row>_c<col>.tif  - 521 feature bands
        CHIP_BASE/<tile>/labels/<tile>_r<row>_c<col>.tif  - 3 bands (Tree, NonTree, NonVeg)
    
    Chips are 64x64 pixels at PACE resolution (~1.85km).
    NoData = -255
    """
    
    CHIP_NO_DATA = -255
    
    suffix = "_chips"
    existing_files = list(OUTPUT_DIR.glob(f"training_data{suffix}_*.parquet"))
    
    if existing_files and not force_extract:
        latest = sorted(existing_files)[-1]
        logger.info(f"Loading existing training data: {latest}")
        return pd.read_parquet(latest)
    
    logger.info("Extracting training data from chip dataset...")
    logger.info(f"  Chip base: {CHIP_BASE}")
    
    # Find all tile directories
    tile_dirs = sorted([d for d in CHIP_BASE.iterdir() if d.is_dir()])
    logger.info(f"  Found {len(tile_dirs)} tile directories")
    
    all_samples = []
    feature_names = None
    
    for tile_dir in tile_dirs:
        tile = tile_dir.name
        inputs_dir = tile_dir / "inputs"
        labels_dir = tile_dir / "labels"
        
        if not inputs_dir.exists() or not labels_dir.exists():
            logger.warning(f"  Skipping {tile}: missing inputs/ or labels/")
            continue
        
        # Find all chip files
        input_files = sorted(inputs_dir.glob("*.tif"))
        
        if len(input_files) == 0:
            logger.warning(f"  Skipping {tile}: no input chips found")
            continue
        
        tile_samples = 0
        
        for input_file in input_files:
            chip_id = input_file.stem  # e.g., "h29v11_r000267_c000068"
            label_file = labels_dir / f"{chip_id}.tif"
            
            if not label_file.exists():
                continue
            
            # Read input features (521 bands)
            with rasterio.open(input_file) as src:
                input_data = src.read()  # Shape: (521, 64, 64)
                
                # Get feature names from band descriptions (first file only)
                if feature_names is None:
                    feature_names = [src.descriptions[i] if src.descriptions[i] else f"Band_{i+1}" 
                                     for i in range(src.count)]
                    logger.info(f"  Found {len(feature_names)} input features")
            
            # Read label (Band 1 = Percent_Tree_Cover)
            with rasterio.open(label_file) as src:
                label_data = src.read(1)  # Shape: (64, 64)
            
            # Extract all valid pixels from this chip
            n_bands, height, width = input_data.shape
            
            for row in range(height):
                for col in range(width):
                    label_val = label_data[row, col]
                    
                    # Skip NoData labels
                    if label_val == CHIP_NO_DATA or label_val < 0 or label_val > 100:
                        continue
                    
                    # Get features for this pixel
                    pixel_features = input_data[:, row, col].astype(np.float32)
                    
                    # Replace NoData with NaN
                    pixel_features[pixel_features == CHIP_NO_DATA] = np.nan
                    
                    # Check if pixel has enough valid features (at least 50%)
                    valid_count = np.sum(np.isfinite(pixel_features))
                    if valid_count < len(pixel_features) * 0.5:
                        continue
                    
                    sample = {
                        'tile': tile,
                        'chip_id': chip_id,
                        'row': row,
                        'col': col,
                        'PercentTree': float(label_val)
                    }
                    sample.update(dict(zip(feature_names, pixel_features)))
                    
                    all_samples.append(sample)
                    tile_samples += 1
        
        logger.info(f"    {tile}: {len(input_files)} chips, {tile_samples:,} valid samples")
    
    if len(all_samples) == 0:
        raise ValueError("No training samples extracted! Check data paths.")
    
    result = pd.DataFrame(all_samples)
    
    logger.info(f"\n  Total samples: {len(result):,}")
    logger.info(f"  Total features: {len(feature_names)}")
    
    # Report NaN statistics
    exclude = ['tile', 'chip_id', 'row', 'col', 'PercentTree']
    metric_cols = [c for c in result.columns if c not in exclude]
    nan_per_feature = result[metric_cols].isna().sum()
    features_with_nan = (nan_per_feature > 0).sum()
    max_nan_pct = (nan_per_feature / len(result) * 100).max() if len(result) > 0 else 0
    
    logger.info(f"  Features with any NaN: {features_with_nan}/{len(metric_cols)}")
    logger.info(f"  Max NaN percentage: {max_nan_pct:.1f}%")
    
    # Show label distribution
    y = result['PercentTree'].values
    logger.info(f"\n  Label distribution:")
    logger.info(f"    Bare (0%):        {np.sum(y == 0):>8,}")
    logger.info(f"    Low (1-25%):      {np.sum((y >= 1) & (y <= 25)):>8,}")
    logger.info(f"    Medium (26-50%):  {np.sum((y >= 26) & (y <= 50)):>8,}")
    logger.info(f"    High (51-80%):    {np.sum((y >= 51) & (y <= 80)):>8,}")
    logger.info(f"    Forest (81-100%): {np.sum((y >= 81) & (y <= 100)):>8,}")
    
    # Save
    save_path = OUTPUT_DIR / f"training_data{suffix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.parquet"
    result.to_parquet(save_path)
    logger.info(f"  Saved: {save_path}")
    
    return result


def apply_balanced_sampling(df, label_col='PercentTree', random_state=42):
    """
    Balanced sampling with configurable class weights.
    """
    np.random.seed(random_state)
    y = df[label_col].values
    
    bare_mask = y == 0
    low_mask = (y >= 1) & (y <= 25)
    med_mask = (y >= 26) & (y <= 50)
    high_mask = (y >= 51) & (y <= 80)
    forest_mask = (y >= 81) & (y <= 100)
    
    n_bare = np.sum(bare_mask)
    n_low = np.sum(low_mask)
    n_med = np.sum(med_mask)
    n_high = np.sum(high_mask)
    n_forest = np.sum(forest_mask)
    
    logger.info(f"Original: bare={n_bare:,}, low={n_low:,}, med={n_med:,}, high={n_high:,}, forest={n_forest:,}")
    
    target_med_high_pct = 1.0 - TARGET_BARE_PCT - TARGET_LOW_PCT - TARGET_FOREST_PCT
    fixed_med_high = n_med + n_high
    estimated_total = fixed_med_high / target_med_high_pct
    
    target_bare = min(int(estimated_total * TARGET_BARE_PCT), n_bare)
    target_low = min(int(estimated_total * TARGET_LOW_PCT), n_low)
    target_forest = int(estimated_total * TARGET_FOREST_PCT)
    
    keep_idx = []
    keep_idx += np.where(med_mask)[0].tolist()
    keep_idx += np.where(high_mask)[0].tolist()
    
    if target_bare <= n_bare:
        keep_idx += np.random.choice(np.where(bare_mask)[0], target_bare, replace=False).tolist()
    else:
        keep_idx += np.where(bare_mask)[0].tolist()
    
    if target_low <= n_low:
        keep_idx += np.random.choice(np.where(low_mask)[0], target_low, replace=False).tolist()
    else:
        keep_idx += np.where(low_mask)[0].tolist()
    
    forest_idx = np.where(forest_mask)[0]
    if target_forest > n_forest:
        keep_idx += np.random.choice(forest_idx, target_forest, replace=True).tolist()
        logger.info(f"  Forest UPSAMPLED: {n_forest:,} → {target_forest:,}")
    elif target_forest < n_forest:
        keep_idx += np.random.choice(forest_idx, target_forest, replace=False).tolist()
    else:
        keep_idx += forest_idx.tolist()
    
    df_balanced = df.iloc[keep_idx].reset_index(drop=True)
    
    y_bal = df_balanced[label_col].values
    n_total = len(y_bal)
    
    final_bare = np.sum(y_bal == 0)
    final_low = np.sum((y_bal >= 1) & (y_bal <= 25))
    final_med = np.sum((y_bal >= 26) & (y_bal <= 50))
    final_high = np.sum((y_bal >= 51) & (y_bal <= 80))
    final_forest = np.sum((y_bal >= 81) & (y_bal <= 100))
    
    logger.info(f"Balanced distribution:")
    logger.info(f"  Bare (0%):       {final_bare:>7,} ({100*final_bare/n_total:>5.1f}%) [target: {TARGET_BARE_PCT*100:.0f}%]")
    logger.info(f"  Low (1-25%):     {final_low:>7,} ({100*final_low/n_total:>5.1f}%) [target: {TARGET_LOW_PCT*100:.0f}%]")
    logger.info(f"  Medium (26-50%): {final_med:>7,} ({100*final_med/n_total:>5.1f}%)")
    logger.info(f"  High (51-80%):   {final_high:>7,} ({100*final_high/n_total:>5.1f}%)")
    logger.info(f"  Forest (81-100%):{final_forest:>7,} ({100*final_forest/n_total:>5.1f}%) [target: {TARGET_FOREST_PCT*100:.0f}%]")
    logger.info(f"Total: {len(df):,} → {n_total:,}")
    
    return df_balanced


def prepare_features(df, include_unsorted=True):
    """
    Prepare features for training.
    
    Since chip inputs already contain all stacked features, we just need to:
    1. Exclude metadata columns
    2. Exclude QA metrics
    3. Optionally exclude UnsortedMonthly features
    """
    exclude = ['tile', 'chip_id', 'row', 'col', 'PercentTree']
    feature_cols = [c for c in df.columns if c not in exclude]
    
    # Exclude QA metrics
    qa_features = [c for c in feature_cols if 'QA_' in c or c.startswith('QA_')]
    feature_cols = [c for c in feature_cols if c not in qa_features]
    if qa_features:
        logger.info(f"  Excluded {len(qa_features)} QA metrics")
    
    # NOTE: AmpBandRefl-Band31 and ThermalGreenBrownDiff-Band31 used to be
    # excluded here as "problematic" -- that was based on them being 0% valid
    # in MODIS_Metrics.tif (fixed in notebook 3l; see skills/PACE-VCF.md).
    # This chip-based pipeline reads its own separately-generated features
    # and never actually had that problem (confirmed both bands were already
    # ~100% valid in the chip files), so the exclusion was needlessly
    # discarding two good features here regardless of the metrics fix.
    
    # Optionally exclude UnsortedMonthly features
    if not include_unsorted:
        unsorted_before = len(feature_cols)
        feature_cols = [c for c in feature_cols if 'UnsortedMonthly' not in c]
        unsorted_removed = unsorted_before - len(feature_cols)
        if unsorted_removed > 0:
            logger.info(f"  Excluded {unsorted_removed} UnsortedMonthly features")
    
    # Categorize features for reporting
    modis_features = [c for c in feature_cols if 'PACE' not in c and 'AltSort' not in c]
    pace_features = [c for c in feature_cols if 'PACE' in c and 'AltSort' not in c]
    altsort_features = [c for c in feature_cols if 'AltSort' in c]
    
    logger.info(f"  Feature counts:")
    logger.info(f"    MODIS-style: {len(modis_features)}")
    logger.info(f"    PACE:        {len(pace_features)}")
    logger.info(f"    AltSort:     {len(altsort_features)}")
    logger.info(f"    Total:       {len(feature_cols)}")
    
    X = df[feature_cols].copy()
    y = df['PercentTree'].values
    
    # Report NaN statistics
    nan_counts = X.isna().sum()
    features_with_nan = (nan_counts > 0).sum()
    total_nan = nan_counts.sum()
    total_values = X.shape[0] * X.shape[1]
    
    logger.info(f"\n  X shape: {X.shape}")
    logger.info(f"  y range: {y.min():.1f}% to {y.max():.1f}%, mean: {y.mean():.1f}%")
    logger.info(f"  NaN handling: NATIVE (XGBoost learns missing value behavior)")
    logger.info(f"    Features with NaN: {features_with_nan}/{len(feature_cols)}")
    logger.info(f"    Total NaN: {total_nan:,} ({100*total_nan/total_values:.2f}%)")
    
    # Show features with most NaN
    if features_with_nan > 0:
        top_nan = nan_counts[nan_counts > 0].sort_values(ascending=False).head(10)
        logger.info(f"    Top NaN features:")
        for feat, count in top_nan.items():
            logger.info(f"      {feat}: {count:,} ({100*count/len(X):.1f}%)")
    
    return X, y, feature_cols


def split_data_70_15_15(X, y, random_state=42):
    """
    Split data into 70% train, 15% validation, 15% test.
    """
    # First split: 70% train, 30% temp
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state
    )
    
    # Second split: 50% of temp for val, 50% for test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=random_state
    )
    
    logger.info(f"\nData split (70-15-15):")
    logger.info(f"  Train:      {len(y_train):>8,} ({100*len(y_train)/len(y):.1f}%)")
    logger.info(f"  Validation: {len(y_val):>8,} ({100*len(y_val)/len(y):.1f}%)")
    logger.info(f"  Test:       {len(y_test):>8,} ({100*len(y_test)/len(y):.1f}%)")
    
    return X_train, X_val, X_test, y_train, y_val, y_test


print("Data loading functions defined ✓")

# =============================================================================
# Cell 3: XGBoost Full Model Functions (UPDATED for 70-15-15 split)
# =============================================================================

def train_xgboost(X_train, X_val, X_test, y_train, y_val, y_test, feature_names):
    """
    Train XGBoost with GPU using all features.
    Uses validation set for early stopping, test set for final evaluation.
    """
    
    logger.info("Training XGBoost (GPU) - Full Model...")
    logger.info(f"  Parameters: {XGB_PARAMS}")
    logger.info(f"  NaN handling: NATIVE (missing=np.nan)")
    
    logger.info(f"  Train: {len(y_train):,}, Val: {len(y_val):,}, Test: {len(y_test):,}")
    
    # Report NaN
    train_nan = X_train.isna().sum().sum()
    val_nan = X_val.isna().sum().sum()
    test_nan = X_test.isna().sum().sum()
    logger.info(f"  Train NaN: {train_nan:,}, Val NaN: {val_nan:,}, Test NaN: {test_nan:,}")
    
    model = xgb.XGBRegressor(**XGB_PARAMS)
    
    # Train with validation set for early stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50
    )
    
    # Evaluate on TEST set (held out completely)
    y_pred_test = model.predict(X_test)
    
    r_pearson, _ = pearsonr(y_test, y_pred_test)
    metrics_test = {
        'r2': r_pearson ** 2,
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'mae': mean_absolute_error(y_test, y_pred_test),
        'best_iteration': model.best_iteration if hasattr(model, 'best_iteration') else XGB_PARAMS['n_estimators']
    }
    
    # Also evaluate on validation set for comparison
    y_pred_val = model.predict(X_val)
    r_pearson_val, _ = pearsonr(y_val, y_pred_val)
    metrics_val = {
        'r2': r_pearson_val ** 2,
        'rmse': np.sqrt(mean_squared_error(y_val, y_pred_val)),
        'mae': mean_absolute_error(y_val, y_pred_val),
    }
    
    logger.info(f"\n  Validation Results:")
    logger.info(f"    R²:   {metrics_val['r2']:.4f}")
    logger.info(f"    RMSE: {metrics_val['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics_val['mae']:.2f}%")
    
    logger.info(f"\n  Test Results (FINAL):")
    logger.info(f"    R²:   {metrics_test['r2']:.4f}")
    logger.info(f"    RMSE: {metrics_test['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics_test['mae']:.2f}%")
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    logger.info(f"\n  Top 20 features:")
    for _, row in importance_df.head(20).iterrows():
        logger.info(f"    {row['feature']:<45} {row['importance']:.4f}")
    
    return model, metrics_test, metrics_val, importance_df, (X_test, y_test, y_pred_test)


def visualize_xgb_results(metrics_test, metrics_val, importance_df, y_test, y_pred):
    """Create visualizations for XGBoost full model."""
    
    logger.info("Creating visualizations...")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].scatter(y_test, y_pred, alpha=0.1, s=5)
    axes[0].plot([0, 100], [0, 100], 'r--', linewidth=2)
    axes[0].set_xlabel('Actual Tree Cover (%)')
    axes[0].set_ylabel('Predicted Tree Cover (%)')
    axes[0].set_title(f'Test Set: Predicted vs Actual\nR²={metrics_test["r2"]:.3f}, RMSE={metrics_test["rmse"]:.1f}%')
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, 100)
    
    top20 = importance_df.head(20)
    axes[1].barh(range(len(top20)), top20['importance'].values)
    axes[1].set_yticks(range(len(top20)))
    axes[1].set_yticklabels(top20['feature'].values, fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Top 20 Feature Importances')
    
    residuals = y_pred - y_test
    axes[2].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    axes[2].axvline(0, color='r', linestyle='--')
    axes[2].set_xlabel('Residual (Predicted - Actual)')
    axes[2].set_ylabel('Count')
    axes[2].set_title(f'Residuals (MAE={metrics_test["mae"]:.2f}%)')
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / "xgb_full_model_results.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"  Saved: {fig_path}")
    plt.show()


def save_xgb_results(model, metrics_test, metrics_val, importance_df, feature_names):
    """Save XGBoost full model results."""
    
    logger.info("Saving results...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    model_path = OUTPUT_DIR / f"pace_vcf_xgb_full_{timestamp}.joblib"
    joblib.dump(model, model_path)
    
    xgb_path = OUTPUT_DIR / f"pace_vcf_xgb_full_{timestamp}.json"
    model.save_model(xgb_path)
    
    importance_df.to_csv(OUTPUT_DIR / f"xgb_full_importance_{timestamp}.csv", index=False)
    
    with open(OUTPUT_DIR / f"xgb_full_metrics_{timestamp}.txt", 'w') as f:
        f.write("PACE-VCF XGBoost Full Model Results\n")
        f.write("="*50 + "\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Split: 70% train / 15% val / 15% test\n")
        f.write(f"\nValidation Set:\n")
        f.write(f"  R²: {metrics_val['r2']:.4f}\n")
        f.write(f"  RMSE: {metrics_val['rmse']:.2f}%\n")
        f.write(f"  MAE: {metrics_val['mae']:.2f}%\n")
        f.write(f"\nTest Set (FINAL):\n")
        f.write(f"  R²: {metrics_test['r2']:.4f}\n")
        f.write(f"  RMSE: {metrics_test['rmse']:.2f}%\n")
        f.write(f"  MAE: {metrics_test['mae']:.2f}%\n")
        f.write(f"\nFeatures: {len(feature_names)}\n")
        f.write(f"NaN handling: NATIVE\n")
    
    logger.info(f"  Model: {model_path}")
    return model_path


print("XGBoost full model functions defined ✓")


# =============================================================================
# Cell 4: Monte Carlo Functions (UPDATED for 70-15-15 split)
# =============================================================================

def run_single_trial(trial_num, X_train, y_train, X_val, y_val, all_features):
    """Run a single Monte Carlo trial using validation set."""
    np.random.seed(RANDOM_STATE + trial_num)
    
    selected_features = np.random.choice(
        all_features, size=min(FEATURES_PER_TRIAL, len(all_features)), replace=False
    )
    
    model = xgb.XGBRegressor(**XGB_TRIAL_PARAMS)
    model.fit(X_train[selected_features], y_train, verbose=False)
    
    y_pred = model.predict(X_val[selected_features])
    
    r_pearson, _ = pearsonr(y_val, y_pred)
    r2 = r_pearson ** 2
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    importances = dict(zip(selected_features, model.feature_importances_))
    
    return {'trial': trial_num, 'r2': r2, 'rmse': rmse, 
            'features': list(selected_features), 'importances': importances}


def run_monte_carlo(X_train, X_val, y_train, y_val, feature_names):
    """
    Run Monte Carlo simulation to find best features.
    Uses validation set for trial evaluation.
    """
    logger.info("\nRunning Monte Carlo simulation...")
    logger.info(f"  NaN handling: NATIVE")
    logger.info(f"  Train: {len(y_train):,}, Val: {len(y_val):,}")
    logger.info(f"  Running {N_TRIALS} trials with {FEATURES_PER_TRIAL} features each...")
    
    all_trials = []
    feature_usage = {f: 0 for f in feature_names}
    feature_importance_sum = {f: 0.0 for f in feature_names}
    feature_importance_count = {f: 0 for f in feature_names}
    
    start_time = time.time()
    
    for trial_num in range(N_TRIALS):
        result = run_single_trial(trial_num, X_train, y_train, X_val, y_val, feature_names)
        all_trials.append(result)
        
        for feat in result['features']:
            feature_usage[feat] += 1
            if feat in result['importances']:
                feature_importance_sum[feat] += result['importances'][feat]
                feature_importance_count[feat] += 1
        
        if (trial_num + 1) % 10 == 0:
            elapsed = time.time() - start_time
            avg_r2 = np.mean([t['r2'] for t in all_trials])
            logger.info(f"    Trial {trial_num + 1}/{N_TRIALS}: avg R²={avg_r2:.4f}, elapsed={elapsed:.1f}s")
    
    logger.info(f"  Completed {N_TRIALS} trials in {time.time() - start_time:.1f}s")
    
    avg_importance = {f: feature_importance_sum[f] / feature_importance_count[f] 
                      if feature_importance_count[f] > 0 else 0.0 for f in feature_names}
    
    trials_df = pd.DataFrame([{'trial': t['trial'], 'r2': t['r2'], 'rmse': t['rmse']} for t in all_trials])
    importance_df = pd.DataFrame([{'feature': f, 'avg_importance': avg_importance[f], 
                                   'usage_count': feature_usage[f]} for f in feature_names]
                                 ).sort_values('avg_importance', ascending=False)
    
    return trials_df, importance_df


def select_top_features(importance_df, min_usage=MIN_FEATURE_USAGE, top_n=TOP_N_FEATURES):
    """Select top-N features that meet minimum usage threshold."""
    logger.info("\nSelecting top features...")
    
    qualified = importance_df[importance_df['usage_count'] >= min_usage].copy()
    logger.info(f"  Features meeting min usage ({min_usage}): {len(qualified)}")
    
    top_features = qualified.head(top_n)['feature'].tolist()
    
    logger.info(f"  Selected top {len(top_features)} features:")
    for _, row in qualified.head(top_n).iterrows():
        logger.info(f"    {row['feature']:<45} imp={row['avg_importance']:.4f}, used={row['usage_count']}")
    
    return top_features


def train_final_mc_model(X_train, X_val, X_test, y_train, y_val, y_test, top_features):
    """
    Train final model with selected features.
    Uses validation for early stopping, test for final evaluation.
    """
    logger.info(f"\nTraining final model with {len(top_features)} features...")
    
    model = xgb.XGBRegressor(**XGB_FINAL_PARAMS)
    model.fit(
        X_train[top_features], y_train,
        eval_set=[(X_train[top_features], y_train), (X_val[top_features], y_val)],
        verbose=50
    )
    
    # Validation metrics
    y_pred_val = model.predict(X_val[top_features])
    r_val, _ = pearsonr(y_val, y_pred_val)
    metrics_val = {
        'r2': r_val ** 2,
        'rmse': np.sqrt(mean_squared_error(y_val, y_pred_val)),
        'mae': mean_absolute_error(y_val, y_pred_val)
    }
    
    # Test metrics (FINAL)
    y_pred_test = model.predict(X_test[top_features])
    r_test, _ = pearsonr(y_test, y_pred_test)
    metrics_test = {
        'r2': r_test ** 2,
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'mae': mean_absolute_error(y_test, y_pred_test)
    }
    
    logger.info(f"\n  Validation Results:")
    logger.info(f"    R²:   {metrics_val['r2']:.4f}")
    logger.info(f"    RMSE: {metrics_val['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics_val['mae']:.2f}%")
    
    logger.info(f"\n  Test Results (FINAL):")
    logger.info(f"    R²:   {metrics_test['r2']:.4f}")
    logger.info(f"    RMSE: {metrics_test['rmse']:.2f}%")
    logger.info(f"    MAE:  {metrics_test['mae']:.2f}%")
    
    final_importance = pd.DataFrame({
        'feature': top_features, 'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    return model, metrics_test, metrics_val, final_importance, (y_test, y_pred_test)


print("Monte Carlo functions defined ✓")


# =============================================================================
# Cell 5: Load and Prepare Data (Run Once)
# =============================================================================

start_time = time.time()

# Load data from chip dataset -- skip reload if df_balanced is already in
# memory from a previous run in this kernel (fast iteration), but don't
# silently break on a fresh kernel where it isn't defined yet.
if 'df_balanced' not in globals():
    df = load_chip_training_data(force_extract=True)
    df_balanced = apply_balanced_sampling(df, label_col='PercentTree')

# Prepare features
X, y, feature_cols = prepare_features(
    df_balanced, 
    include_unsorted=INCLUDE_UNSORTED
)

# NEW: 70-15-15 split
X_train, X_val, X_test, y_train, y_val, y_test = split_data_70_15_15(X, y, random_state=RANDOM_STATE)

# Summary
print(f"\n{'='*60}")
print("FEATURE SUMMARY")
print(f"{'='*60}")

modis_style = [c for c in feature_cols if not c.startswith('PACE_') and not c.startswith('AltSort_')]
print(f"  MODIS-style features: {len(modis_style)}")
print(f"  PACE features: {len([c for c in feature_cols if c.startswith('PACE_')])}")
print(f"  AltSort features: {len([c for c in feature_cols if c.startswith('AltSort_')])}")
print(f"  TOTAL: {len(feature_cols)}")
print(f"{'='*60}")
print(f"  NaN handling: NATIVE")
print(f"{'='*60}")

print(f"\nData prepared in {time.time() - start_time:.1f}s")
print(f"  Total samples: {len(y):,}")
print(f"  Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")
print(f"  Features: {len(feature_cols)}")


# =============================================================================
# Cell 6: Train Full Model
# =============================================================================

model, metrics_test, metrics_val, importance_df, test_data = train_xgboost(
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols
)

X_test_out, y_test_out, y_pred = test_data
visualize_xgb_results(metrics_test, metrics_val, importance_df, y_test_out, y_pred)
save_xgb_results(model, metrics_test, metrics_val, importance_df, feature_cols)


# =============================================================================
# Cell 7: Monte Carlo Feature Selection
# =============================================================================

trials_df, mc_importance_df = run_monte_carlo(X_train, X_val, y_train, y_val, feature_cols)
top_features = select_top_features(mc_importance_df)

mc_model, mc_metrics_test, mc_metrics_val, mc_final_importance, mc_test_data = train_final_mc_model(
    X_train, X_val, X_test, y_train, y_val, y_test, top_features
)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Full model test R²: {metrics_test['r2']:.4f}")
print(f"MC model test R²:   {mc_metrics_test['r2']:.4f}")
print("="*60)

In [ ]:
# =============================================================================
# Save Monte Carlo Results
# =============================================================================

def save_mc_results(model, metrics_test, metrics_val, trials_df, importance_df, final_importance, top_features):
    """Save Monte Carlo results."""
    logger.info("\nSaving Monte Carlo results...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save model
    model_path = OUTPUT_DIR / f"pace_vcf_mc_xgb_{timestamp}.json"
    model.save_model(model_path)
    
    joblib_path = OUTPUT_DIR / f"pace_vcf_mc_xgb_{timestamp}.joblib"
    joblib.dump(model, joblib_path)
    
    # Save trial results
    trials_df.to_csv(OUTPUT_DIR / f"mc_trials_{timestamp}.csv", index=False)
    importance_df.to_csv(OUTPUT_DIR / f"mc_all_importance_{timestamp}.csv", index=False)
    final_importance.to_csv(OUTPUT_DIR / f"mc_final_importance_{timestamp}.csv", index=False)
    
    # Save top features list
    with open(OUTPUT_DIR / f"mc_top_features_{timestamp}.txt", 'w') as f:
        for feat in top_features:
            f.write(f"{feat}\n")
    
    # Save metrics summary
    with open(OUTPUT_DIR / f"mc_metrics_{timestamp}.txt", 'w') as f:
        f.write("PACE-VCF Monte Carlo Feature Selection Results\n")
        f.write("="*60 + "\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"N_trials: {N_TRIALS}\n")
        f.write(f"Features_per_trial: {FEATURES_PER_TRIAL}\n")
        f.write(f"Top_N_features: {TOP_N_FEATURES}\n")
        f.write(f"NaN handling: NATIVE\n")
        f.write(f"\nValidation Set:\n")
        f.write(f"  R²:   {metrics_val['r2']:.4f}\n")
        f.write(f"  RMSE: {metrics_val['rmse']:.2f}%\n")
        f.write(f"  MAE:  {metrics_val['mae']:.2f}%\n")
        f.write(f"\nTest Set (FINAL):\n")
        f.write(f"  R²:   {metrics_test['r2']:.4f}\n")
        f.write(f"  RMSE: {metrics_test['rmse']:.2f}%\n")
        f.write(f"  MAE:  {metrics_test['mae']:.2f}%\n")
        f.write(f"\nTop {len(top_features)} Features:\n")
        for i, feat in enumerate(top_features, 1):
            f.write(f"  {i:2}. {feat}\n")
    
    logger.info(f"  Model: {model_path}")
    logger.info(f"  Joblib: {joblib_path}")
    logger.info(f"  Top features: {OUTPUT_DIR / f'mc_top_features_{timestamp}.txt'}")
    
    return model_path

# Save the MC model (assuming variables are still in memory)
save_mc_results(
    mc_model, 
    mc_metrics_test, 
    mc_metrics_val, 
    trials_df, 
    mc_importance_df, 
    mc_final_importance, 
    top_features
)

In [ ]:
# =============================================================================
# Visualization: Scatter Plots for Full and MC Models
# =============================================================================

def plot_model_comparison(y_test, y_pred_full, y_pred_mc, metrics_full, metrics_mc):
    """Create scatter plots comparing Full and MC models."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Full Model
    axes[0].scatter(y_test, y_pred_full, alpha=0.1, s=5, c='blue')
    axes[0].plot([0, 100], [0, 100], 'r--', linewidth=2, label='1:1 line')
    axes[0].set_xlabel('Actual Tree Cover (%)', fontsize=12)
    axes[0].set_ylabel('Predicted Tree Cover (%)', fontsize=12)
    axes[0].set_title(f'Full Model (519 features)\nR²={metrics_full["r2"]:.4f}, RMSE={metrics_full["rmse"]:.2f}%', fontsize=12)
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, 100)
    axes[0].set_aspect('equal')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: MC Model (50 features)
    axes[1].scatter(y_test, y_pred_mc, alpha=0.1, s=5, c='green')
    axes[1].plot([0, 100], [0, 100], 'r--', linewidth=2, label='1:1 line')
    axes[1].set_xlabel('Actual Tree Cover (%)', fontsize=12)
    axes[1].set_ylabel('Predicted Tree Cover (%)', fontsize=12)
    axes[1].set_title(f'MC Model (50 features)\nR²={metrics_mc["r2"]:.4f}, RMSE={metrics_mc["rmse"]:.2f}%', fontsize=12)
    axes[1].set_xlim(0, 100)
    axes[1].set_ylim(0, 100)
    axes[1].set_aspect('equal')
    axes[1].legend(loc='upper left')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Residual comparison
    residuals_full = y_pred_full - y_test
    residuals_mc = y_pred_mc - y_test
    
    axes[2].hist(residuals_full, bins=50, alpha=0.5, label=f'Full (MAE={metrics_full["mae"]:.2f}%)', color='blue', edgecolor='darkblue')
    axes[2].hist(residuals_mc, bins=50, alpha=0.5, label=f'MC (MAE={metrics_mc["mae"]:.2f}%)', color='green', edgecolor='darkgreen')
    axes[2].axvline(0, color='r', linestyle='--', linewidth=2)
    axes[2].set_xlabel('Residual (Predicted - Actual)', fontsize=12)
    axes[2].set_ylabel('Count', fontsize=12)
    axes[2].set_title('Residual Distribution Comparison', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / "model_comparison_scatter.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"Saved: {fig_path}")
    plt.show()
    
    return fig


def plot_detailed_scatter(y_test, y_pred, model_name, metrics, color='blue'):
    """Create detailed scatter plot with density and marginal histograms."""
    
    fig = plt.figure(figsize=(10, 10))
    
    # Main scatter plot
    gs = fig.add_gridspec(4, 4)
    ax_main = fig.add_subplot(gs[1:4, 0:3])
    ax_top = fig.add_subplot(gs[0, 0:3], sharex=ax_main)
    ax_right = fig.add_subplot(gs[1:4, 3], sharey=ax_main)
    
    # Scatter with hexbin for density
    hb = ax_main.hexbin(y_test, y_pred, gridsize=50, cmap='Blues', mincnt=1)
    ax_main.plot([0, 100], [0, 100], 'r--', linewidth=2, label='1:1 line')
    ax_main.set_xlabel('Actual Tree Cover (%)', fontsize=12)
    ax_main.set_ylabel('Predicted Tree Cover (%)', fontsize=12)
    ax_main.set_xlim(0, 100)
    ax_main.set_ylim(0, 100)
    ax_main.set_aspect('equal')
    ax_main.legend(loc='upper left')
    ax_main.grid(True, alpha=0.3)
    
    # Colorbar
    cb = plt.colorbar(hb, ax=ax_main, shrink=0.6, label='Count')
    
    # Top histogram (actual values)
    ax_top.hist(y_test, bins=50, color=color, alpha=0.7, edgecolor='black')
    ax_top.set_ylabel('Count')
    ax_top.set_title(f'{model_name}\nR²={metrics["r2"]:.4f}, RMSE={metrics["rmse"]:.2f}%, MAE={metrics["mae"]:.2f}%', fontsize=14)
    plt.setp(ax_top.get_xticklabels(), visible=False)
    
    # Right histogram (predicted values)
    ax_right.hist(y_pred, bins=50, orientation='horizontal', color=color, alpha=0.7, edgecolor='black')
    ax_right.set_xlabel('Count')
    plt.setp(ax_right.get_yticklabels(), visible=False)
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f"{model_name.lower().replace(' ', '_')}_detailed_scatter.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"Saved: {fig_path}")
    plt.show()
    
    return fig


def plot_residuals_by_class(y_test, y_pred, model_name, metrics):
    """Plot residuals broken down by tree cover class."""
    
    residuals = y_pred - y_test
    
    # Define classes
    classes = [
        ('Bare (0%)', y_test == 0),
        ('Low (1-25%)', (y_test >= 1) & (y_test <= 25)),
        ('Medium (26-50%)', (y_test >= 26) & (y_test <= 50)),
        ('High (51-80%)', (y_test >= 51) & (y_test <= 80)),
        ('Forest (81-100%)', (y_test >= 81) & (y_test <= 100))
    ]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot of residuals by class
    class_residuals = [residuals[mask] for _, mask in classes]
    class_names = [name for name, _ in classes]
    
    bp = axes[0].boxplot(class_residuals, labels=class_names, patch_artist=True)
    colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_ylabel('Residual (Predicted - Actual)', fontsize=12)
    axes[0].set_title(f'{model_name}: Residuals by Tree Cover Class', fontsize=12)
    axes[0].tick_params(axis='x', rotation=15)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # MAE by class
    class_mae = [np.mean(np.abs(residuals[mask])) for _, mask in classes]
    class_counts = [np.sum(mask) for _, mask in classes]
    
    bars = axes[1].bar(class_names, class_mae, color=colors, alpha=0.7, edgecolor='black')
    axes[1].set_ylabel('Mean Absolute Error (%)', fontsize=12)
    axes[1].set_title(f'{model_name}: MAE by Tree Cover Class', fontsize=12)
    axes[1].tick_params(axis='x', rotation=15)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar, count, mae in zip(bars, class_counts, class_mae):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
                     f'n={count:,}\nMAE={mae:.2f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    fig_path = OUTPUT_DIR / f"{model_name.lower().replace(' ', '_')}_residuals_by_class.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    logger.info(f"Saved: {fig_path}")
    plt.show()
    
    return fig


# =============================================================================
# Generate all plots
# =============================================================================

# Get predictions (if not already in memory)
# Full model predictions
y_pred_full = model.predict(X_test)

# MC model predictions
y_pred_mc = mc_model.predict(X_test[top_features])

# 1. Side-by-side comparison
print("="*60)
print("GENERATING SCATTER PLOTS")
print("="*60)

plot_model_comparison(y_test, y_pred_full, y_pred_mc, metrics_test, mc_metrics_test)

# 2. Detailed scatter for Full model
plot_detailed_scatter(y_test, y_pred_full, "Full Model (519 features)", metrics_test, color='blue')

# 3. Detailed scatter for MC model
plot_detailed_scatter(y_test, y_pred_mc, "MC Model (50 features)", mc_metrics_test, color='green')

# 4. Residuals by class for both models
plot_residuals_by_class(y_test, y_pred_full, "Full Model", metrics_test)
plot_residuals_by_class(y_test, y_pred_mc, "MC Model", mc_metrics_test)

print("\n" + "="*60)
print("ALL PLOTS SAVED TO:", OUTPUT_DIR)
print("="*60)

In [ ]:
# # Check if thermal/snow features are in the final feature_cols
# thermal_in_features = [c for c in feature_cols if 'LST' in c or 'Band31' in c or 'Thermal' in c]
# snow_in_features = [c for c in feature_cols if 'Snow' in c]

# print(f"Thermal features in training: {len(thermal_in_features)}")
# for t in thermal_in_features:
#     print(f"  - {t}")

# print(f"\nSnow features in training: {len(snow_in_features)}")
# for s in snow_in_features:
#     print(f"  - {s}")

# print(f"\nTotal features for training: {len(feature_cols)}")